In [ ]:
!pip install gin-config polars einops tqdm gdown -q
!pip install torch-geometric -q

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '')
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q

!git clone https://github.com/mikhaildanilov/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article.git
%cd deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article
!gdown 1qGxgmx7G_WB7JE4Cn_bEcZ_o_NAJLE3G -O P5_data.zip
!unzip -q P5_data.zip && mkdir -p dataset/amazon/raw && mv data/* dataset/amazon/raw/

In [ ]:
import shutil, os
os.makedirs('dataset/amazon/processed', exist_ok=True)
shutil.copy('/kaggle/input/datasets/<YOUR PATH>/data_beauty.pt',
            'dataset/amazon/processed/data_beauty.pt')
print("OK:", os.path.exists('dataset/amazon/processed/data_beauty.pt'))

In [ ]:
!ls trained_models/rqvae_amazon_beauty/

In [ ]:
%%bash
cd /kaggle/working/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article
git checkout -- data/
ls data/

In [ ]:
%%bash
cd /kaggle/working/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article
sed -i 's/train.vae_codebook_size=256/train.vae_codebook_size=[256,256,256]/' configs/decoder_amazon.gin
sed -i 's/num_embeddings_per_hierarchy=vae_codebook_size,/num_embeddings_per_hierarchy=max(vae_codebook_size) if isinstance(vae_codebook_size, list) else vae_codebook_size,/' train_decoder.py
sed -i 's/train.full_eval_every=1000/train.full_eval_every=10000/' configs/decoder_amazon.gin
grep "vae_codebook_size" configs/decoder_amazon.gin
grep "num_embeddings_per_hierarchy" train_decoder.py

In [ ]:
# %%bash
!cd /kaggle/working/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article
!WANDB_API_KEY="<YOUR WANDB API KEY>" python train_decoder.py configs/decoder_amazon.gin